# Figure S1 — corona-charge calibration and semiconductor band bending

This notebook reproduces **Figure S1** and quantifies the difference between a dielectric-only corona-voltage conversion and a self-consistent conversion that includes the InP surface potential.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator, MaxNLocator

# Locate the local SURPASS source tree from either the project root or this notebook directory.
root = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
             if (p / "src" / "surpass").exists()), None)
if root is None:
    raise FileNotFoundError("Could not locate the SURPASS project root.")
sys.path.insert(0, str(root / "src"))

from surpass import BulkModel, Wafer, InterfaceDefectModel, SurfaceSRHModel
from surpass.constants import EPS0_F_CM, Q_C

try:
    import scienceplots
    plt.style.use(["science", "notebook"])
except ImportError:
    pass

plt.rcParams.update({
    "font.size": 12,
    "axes.labelsize": 13,
    "axes.titlesize": 13,
    "legend.fontsize": 11,
    "axes.spines.top": True,
    "axes.spines.right": True,
    "figure.dpi": 130,
    "savefig.dpi": 400,
})

## 1. User-adjustable parameters

In [ ]:
# Wafer parameters used in the paper
ND_N_INP_CM3 = 8.5e18
NA_P_INP_CM3 = 5.1e18
TEMPERATURE_K = 300.0

# The initial net effective charge is deliberately varied rather than assumed.
# It can include dielectric fixed charge and charged interface states.
Q_INITIAL_MIN_CM2 = -2.0e12
Q_INITIAL_MAX_CM2 = +4.0e12
N_INITIAL_CHARGE_POINTS = 121

# Negative deposited corona-charge range; x below is its positive magnitude.
MAX_NEGATIVE_CORONA_CM2 = 6.0e12
N_CHARGE_POINTS = 241

# Replace the thicknesses below by the experimentally measured values.
POX_THICKNESS_NM = 10.0
ALOX_THICKNESS_NM = 10.0
STACK_THICKNESS_NM = POX_THICKNESS_NM + ALOX_THICKNESS_NM
STACK_EPS_R = 7.4

# A very small injection is used to evaluate near-equilibrium electrostatics.
# Setting interface_charge=None isolates the SCR contribution.
DELTA_N_ELECTROSTATICS_CM3 = 1.0e8
ENABLE_BGN = False

C_DIEL_F_CM2 = (
    EPS0_F_CM * STACK_EPS_R / (STACK_THICKNESS_NM * 1e-7)
)

print(f"Stack thickness       = {STACK_THICKNESS_NM:.1f} nm")
print(f"Dielectric capacitance = {C_DIEL_F_CM2:.3e} F cm^-2")
print(
    "Dielectric voltage per 1e12 cm^-2 = "
    f"{Q_C*1e12/C_DIEL_F_CM2:.3f} V"
)

## 2. Self-consistent InP surface potential

The surface potential follows from charge neutrality,

$$
q(Q_{f,0}+Q_c)+Q_{\mathrm{sc}}(\psi_s)=0,
$$

where $Q_{\mathrm{sc}}$ is obtained by integrating Poisson's equation with Fermi--Dirac carrier populations. The calculation below deliberately excludes $Q_{it}$; including it would require assumptions about the interface-state distribution and charge-neutrality level.

In [ ]:
def make_bulk_model(polarity):
    if polarity == "n":
        wafer = Wafer(
            "InP", donor_cm3=ND_N_INP_CM3,
            thickness_um=500.0, temperature_k=TEMPERATURE_K,
            label="n-InP",
        )
    elif polarity == "p":
        wafer = Wafer(
            "InP", acceptor_cm3=NA_P_INP_CM3,
            thickness_um=500.0, temperature_k=TEMPERATURE_K,
            label="p-InP",
        )
    else:
        raise ValueError("polarity must be 'n' or 'p'")
    return BulkModel(wafer, enable_bgn=ENABLE_BGN)


# The defect object is required by SurfaceSRHModel, but interface-state charge
# is explicitly disabled. Therefore its Dit and capture parameters do not
# affect the equilibrium charge balance used here.
defects = InterfaceDefectModel()
bulk_models = {pol: make_bulk_model(pol) for pol in ("n", "p")}


def surface_potential_v(polarity, net_external_charge_cm2):
    model = SurfaceSRHModel(
        bulk_models[polarity],
        qf_number_cm2=float(net_external_charge_cm2),
        defects=defects,
        electrostatics="self_consistent",
        interface_charge=None,
    )
    state = model.state(DELTA_N_ELECTROSTATICS_CM3)
    return state.psi_surface_v


negative_corona_magnitude_cm2 = np.linspace(
    0.0, MAX_NEGATIVE_CORONA_CM2, N_CHARGE_POINTS
)
initial_effective_charge_cm2 = np.linspace(
    Q_INITIAL_MIN_CM2, Q_INITIAL_MAX_CM2, N_INITIAL_CHARGE_POINTS
)

# Rows correspond to initial effective charge; columns correspond to deposited
# negative-corona magnitude. The actual signed corona charge is -|Nc|.
Q_CORONA_MAG, Q_INITIAL = np.meshgrid(
    negative_corona_magnitude_cm2, initial_effective_charge_cm2
)
Q_CORONA_SIGNED = -Q_CORONA_MAG
Q_NET = Q_INITIAL + Q_CORONA_SIGNED

results = {}
for polarity in ("n", "p"):
    # Cache psi_s over the union of required net-charge values. Interpolation
    # is both faster and smoother than solving every two-dimensional pixel.
    q_lookup = np.linspace(
        Q_INITIAL_MIN_CM2 - MAX_NEGATIVE_CORONA_CM2,
        Q_INITIAL_MAX_CM2,
        1201,
    )
    psi_lookup = np.array([
        surface_potential_v(polarity, q) for q in q_lookup
    ])
    psi_initial = np.interp(initial_effective_charge_cm2, q_lookup, psi_lookup)
    psi_final = np.interp(Q_NET.ravel(), q_lookup, psi_lookup).reshape(Q_NET.shape)
    delta_psi = psi_final - psi_initial[:, None]

    delta_v_dielectric = Q_C*Q_CORONA_SIGNED/C_DIEL_F_CM2
    delta_v_exact = delta_v_dielectric + delta_psi
    q_apparent_signed = C_DIEL_F_CM2*delta_v_exact/Q_C

    # The relative error is undefined at zero applied corona charge.
    relative_error_percent = np.full_like(Q_CORONA_MAG, np.nan)
    nonzero = Q_CORONA_MAG > 0
    relative_error_percent[nonzero] = 100.0*(
        np.abs(q_apparent_signed[nonzero])/Q_CORONA_MAG[nonzero] - 1.0
    )

    results[polarity] = dict(
        psi_initial=psi_initial,
        psi_final=psi_final,
        delta_psi=delta_psi,
        delta_v_exact=delta_v_exact,
        q_apparent_signed=q_apparent_signed,
        relative_error_percent=relative_error_percent,
    )

print(
    "Initial effective-charge range: "
    f"{Q_INITIAL_MIN_CM2/1e12:+.1f} to {Q_INITIAL_MAX_CM2/1e12:+.1f} "
    "x 1e12 cm^-2"
)

## 3. Error in the dielectric-only conversion

In [ ]:
for polarity in ("n", "p"):
    error = results[polarity]["relative_error_percent"]
    print(
        f"{polarity}-InP error over the plotted nonzero-charge domain: "
        f"{np.nanmin(error):.1f}% to {np.nanmax(error):.1f}%"
    )

# A neutral initial state is a useful non-circular reference, not an assertion
# about the actual passivated sample.
i_neutral = np.argmin(np.abs(initial_effective_charge_cm2))
i_max_corona = -1
for polarity in ("n", "p"):
    error_at_corner = results[polarity]["relative_error_percent"][
        i_neutral, i_max_corona
    ]
    print(
        f"{polarity}-InP at initial charge = 0 and -6e12 cm^-2 corona: "
        f"{error_at_corner:.1f}%"
    )

## 4. Figure for the Supplementary Information

The initial effective charge is not known independently and is therefore shown as a sensitivity axis rather than fixed a priori. The color gives the relative overestimate in the magnitude of deposited negative corona charge when the complete Kelvin-probe voltage change is converted using only the dielectric capacitance. The horizontal dashed line marks a neutral initial effective charge for reference; it is not an assumption about the experiment.

In [ ]:
x = negative_corona_magnitude_cm2/1e12
y = initial_effective_charge_cm2/1e12
X, Y = np.meshgrid(x, y)

fig, axes = plt.subplots(1, 2, figsize=(10.8, 4.8), sharex=True, sharey=True)

# Shared scale emphasizes that the two doping polarities are compared on the
# same quantitative basis. Extend the final interval if parameters are changed.
combined_max = max(
    np.nanmax(results["n"]["relative_error_percent"]),
    np.nanmax(results["p"]["relative_error_percent"]),
)
upper = max(22.0, np.ceil(combined_max/2.0)*2.0)
levels = np.linspace(0.0, upper, 23)

for ax, polarity, title in zip(axes, ("n", "p"), ("n-InP", "p-InP")):
    cf = ax.contourf(
        X, Y, results[polarity]["relative_error_percent"],
        levels=levels, cmap="viridis", extend="max",
    )
    ax.axhline(0.0, color="white", lw=1.6, ls="--")
    ax.text(
        0.18, 0.18, r"$N_{\mathrm{eff},0}=0$",
        color="black", fontsize=11,
        bbox=dict(boxstyle="round,pad=0.22", fc="white", ec="0.4", alpha=0.82),
    )
    ax.set_title(title)
    ax.set_xlim(0, MAX_NEGATIVE_CORONA_CM2/1e12)
    ax.set_ylim(Q_INITIAL_MIN_CM2/1e12, Q_INITIAL_MAX_CM2/1e12)
    ax.xaxis.set_major_locator(MultipleLocator(1.0))
    ax.yaxis.set_major_locator(MultipleLocator(1.0))
    ax.tick_params(direction="in", top=True, right=True)

axes[0].set_ylabel(
    r"Initial effective charge, $N_{\mathrm{eff},0}$ "
    r"($10^{12}$ cm$^{-2}$)"
)

cax = fig.add_axes([0.885, 0.20, 0.025, 0.64])
cbar = fig.colorbar(cf, cax=cax)
cbar.set_label("Charge overestimate (%)")
cbar.locator = MaxNLocator(nbins=7, integer=True)
cbar.update_ticks()

fig.supxlabel(
    r"Deposited negative-corona magnitude, $-N_c$ "
    r"($10^{12}$ cm$^{-2}$)",
    y=0.035,
)

fig.suptitle(
    rf"SCR contribution to corona-charge calibration "
    rf"($t_{{\mathrm{{diel}}}}={STACK_THICKNESS_NM:.0f}$ nm, "
    rf"$\varepsilon_r={STACK_EPS_R:.1f}$)",
    y=0.99,
)
fig.subplots_adjust(top=0.84, bottom=0.20, left=0.10, right=0.84, wspace=0.10)

fig.savefig("inp_corona_scr_correction.png", bbox_inches="tight")
fig.savefig("inp_corona_scr_correction.pdf", bbox_inches="tight")
plt.show()

## Interpretation

For the default 20 nm, $\varepsilon_r=7.4$ stack, including the SCR makes the magnitude of the Kelvin-probe voltage change larger than the dielectric-only value. Consequently, converting the complete voltage with $C_{\mathrm{diel}}\Delta V/q$ overestimates the magnitude of deposited negative corona charge. The correction depends on the unknown initial effective charge, as well as stack thickness, permittivity, wafer doping, temperature, injection, and interface-state charging.

The figure is therefore a sensitivity analysis rather than an absolute recalibration. It avoids presupposing the positive charge that the combined experiment and modelling are intended to support. The calculation should be repeated after replacing the illustrative stack thickness by its measured value.